[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module3/09-data-formats.ipynb)

# Module 3.9 — Data Formats: YAML, TOML, and XML
**Module 3: Automation & Scripting** | Estimated time: 20 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Read and write YAML files with `PyYAML` (`safe_load` / `safe_dump`)
- Use YAML multi-document files, anchors, and aliases
- Parse TOML configuration files using Python 3.11's built-in `tomllib`
- Parse and create XML documents with `xml.etree.ElementTree`
- Handle XML namespaces
- Choose the right format (JSON / YAML / TOML / XML) for a given use case

In [ ]:
!pip install pyyaml -q

import yaml
import tomllib          # stdlib in Python 3.11+
import xml.etree.ElementTree as ET
import json
from pathlib import Path
from pprint import pprint

OUTDIR = Path('/tmp/pypath_formats')
OUTDIR.mkdir(exist_ok=True)

print('PyYAML version:', yaml.__version__)
print('tomllib version: stdlib (3.11+)')
print('xml.etree.ElementTree: stdlib')

## 1. YAML — Human-Friendly Configuration

YAML (YAML Ain't Markup Language) is commonly used for configuration files, CI/CD pipelines (GitHub Actions, Docker Compose), and Kubernetes manifests.

**Always use `yaml.safe_load` / `yaml.safe_dump`** — never `yaml.load()` without a Loader, as it can execute arbitrary Python.

In [ ]:
# Write a YAML config file
yaml_content = """
app:
  name: PyPath Scraper
  version: "1.4.2"
  debug: false

database:
  host: localhost
  port: 5432
  name: pypath_db
  pool_size: 10

scraping:
  rate_limit_seconds: 1.5
  max_retries: 3
  user_agent: "PyPath/1.4.2 (+https://pypath.dev)"
  allowed_domains:
    - books.toscrape.com
    - quotes.toscrape.com
  excluded_paths:
    - /admin
    - /private

logging:
  level: INFO
  file: /var/log/pypath.log
  max_bytes: 10485760   # 10 MB
  backup_count: 5
"""

cfg_path = OUTDIR / 'config.yaml'
cfg_path.write_text(yaml_content)
print(f'Written: {cfg_path}')

# Read it back
with cfg_path.open() as f:
    config = yaml.safe_load(f)

print('\nParsed config:')
pprint(config)

In [ ]:
# Accessing nested values
print('App name     :', config['app']['name'])
print('DB port      :', config['database']['port'])
print('Rate limit   :', config['scraping']['rate_limit_seconds'])
print('Allowed hosts:', config['scraping']['allowed_domains'])
print('Log level    :', config['logging']['level'])
print()

# Modifying and writing back
config['app']['version'] = '1.4.3'
config['scraping']['rate_limit_seconds'] = 2.0

with (OUTDIR / 'config_updated.yaml').open('w') as f:
    yaml.safe_dump(config, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print('Updated YAML written.')
print((OUTDIR / 'config_updated.yaml').read_text())

## 2. YAML Multi-Document Files and Anchors/Aliases

YAML supports multiple documents in a single file (separated by `---`) and reusable values via anchors (`&name`) and aliases (`*name`).

In [ ]:
multi_doc_yaml = """
# Anchors define reusable values
defaults: &defaults
  retries: 3
  timeout: 30
  encoding: utf-8

---
job_id: books_job
<<: *defaults          # merge all defaults fields in
url: https://books.toscrape.com/
max_pages: 50

---
job_id: quotes_job
<<: *defaults
url: https://quotes.toscrape.com/
max_pages: 10
timeout: 60            # override the default timeout
"""

# yaml.safe_load_all yields each document
docs = list(yaml.safe_load_all(multi_doc_yaml))
print(f'Documents in multi-doc file: {len(docs)}')
for i, doc in enumerate(docs):
    if doc:
        print(f'\n--- Document {i} ---')
        pprint(doc)

## 3. TOML — Configuration for Python Projects

`tomllib` was added to the standard library in Python 3.11. It reads TOML files (the format used by `pyproject.toml`). For writing TOML you need the third-party `tomli-w` package.

In [ ]:
%%writefile /tmp/pypath_formats/pyproject_sample.toml
[project]
name = "pypath-scraper"
version = "1.4.2"
description = "A professional web scraping toolkit"
requires-python = ">=3.11"

[project.dependencies]
python = "^3.11"
requests = ">=2.28"
beautifulsoup4 = ">=4.11"
apscheduler = ">=3.10"

[project.scripts]
pypath = "pypath.cli:main"

[tool.scraper]
rate_limit = 1.5
max_retries = 3
default_timeout = 30
allowed_domains = ["books.toscrape.com", "quotes.toscrape.com"]

[tool.scraper.headers]
User-Agent = "PyPath/1.4.2"
Accept = "text/html,application/xhtml+xml"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

In [ ]:
toml_path = OUTDIR / 'pyproject_sample.toml'

# tomllib requires binary mode
with toml_path.open('rb') as f:
    pyproject = tomllib.load(f)

print('Parsed TOML:')
pprint(pyproject)
print()

# Access values
print('Project name    :', pyproject['project']['name'])
print('Version         :', pyproject['project']['version'])
print('Python required :', pyproject['project']['requires-python'])
print('Rate limit      :', pyproject['tool']['scraper']['rate_limit'])
print('User-Agent      :', pyproject['tool']['scraper']['headers']['User-Agent'])
print('Allowed domains :', pyproject['tool']['scraper']['allowed_domains'])

In [ ]:
# tomllib.loads() parses a TOML string directly
toml_string = """
[server]
host = "0.0.0.0"
port = 8080
workers = 4

[server.tls]
enabled = true
cert = "/etc/ssl/cert.pem"
key  = "/etc/ssl/key.pem"

[jobs]
schedules = ["0 9 * * 1-5", "0 18 * * 1-5"]
max_concurrent = 3
"""

server_cfg = tomllib.loads(toml_string)
print('Server config:', server_cfg['server'])
print('TLS enabled  :', server_cfg['server']['tls']['enabled'])
print('Schedules    :', server_cfg['jobs']['schedules'])

## 4. XML — `xml.etree.ElementTree`

XML is common in enterprise systems, RSS feeds, SOAP APIs, and SVG files. Python's `xml.etree.ElementTree` is a fast, lightweight parser built into the standard library.

In [ ]:
xml_string = """
<?xml version="1.0" encoding="UTF-8"?>
<catalog version="2024-03">
  <book id="bk101" available="true">
    <title>A Light in the Attic</title>
    <author>Shel Silverstein</author>
    <price currency="GBP">51.77</price>
    <rating>3</rating>
    <genres>
      <genre>Poetry</genre>
      <genre>Children</genre>
    </genres>
  </book>
  <book id="bk102" available="true">
    <title>Tipping the Velvet</title>
    <author>Sarah Waters</author>
    <price currency="GBP">53.74</price>
    <rating>1</rating>
    <genres>
      <genre>Historical Fiction</genre>
    </genres>
  </book>
  <book id="bk103" available="false">
    <title>Soumission</title>
    <author>Michel Houellebecq</author>
    <price currency="EUR">50.10</price>
    <rating>1</rating>
    <genres>
      <genre>Fiction</genre>
    </genres>
  </book>
</catalog>
"""

root = ET.fromstring(xml_string)
print('Root tag     :', root.tag)
print('Root attribs :', root.attrib)
print('Child count  :', len(root))

# Iterate all book elements
for book in root.findall('book'):
    book_id   = book.get('id')
    available = book.get('available')
    title     = book.findtext('title')
    author    = book.findtext('author')
    price_el  = book.find('price')
    price     = price_el.text
    currency  = price_el.get('currency')
    rating    = book.findtext('rating')
    genres    = [g.text for g in book.findall('genres/genre')]
    print(f'\n[{book_id}] {title}')
    print(f'  Author   : {author}')
    print(f'  Price    : {price} {currency}')
    print(f'  Rating   : {rating}/5')
    print(f'  Available: {available}')
    print(f'  Genres   : {genres}')

## 5. Creating and Writing XML

In [ ]:
# Build an XML document programmatically
def create_report_xml(jobs: list[dict]) -> ET.Element:
    """Create a scraping report XML tree."""
    report = ET.Element('report')
    report.set('generated', str(__import__('datetime').date.today()))
    report.set('version', '1')

    summary = ET.SubElement(report, 'summary')
    ET.SubElement(summary, 'total_jobs').text  = str(len(jobs))
    ET.SubElement(summary, 'successful').text  = str(sum(1 for j in jobs if j['status'] == 'done'))
    ET.SubElement(summary, 'failed').text      = str(sum(1 for j in jobs if j['status'] == 'failed'))

    jobs_el = ET.SubElement(report, 'jobs')
    for job in jobs:
        job_el = ET.SubElement(jobs_el, 'job')
        job_el.set('id',     job['id'])
        job_el.set('status', job['status'])
        ET.SubElement(job_el, 'name').text      = job['name']
        ET.SubElement(job_el, 'pages').text     = str(job['pages'])
        ET.SubElement(job_el, 'items').text     = str(job['items'])
        ET.SubElement(job_el, 'duration').text  = f"{job['duration']:.1f}s"
        if 'error' in job:
            ET.SubElement(job_el, 'error').text = job['error']
    return report


jobs_data = [
    {'id': '1', 'name': 'books_catalogue',  'status': 'done',   'pages': 50, 'items': 1000, 'duration': 62.5},
    {'id': '2', 'name': 'quotes_scraper',   'status': 'done',   'pages': 10, 'items': 100,  'duration': 15.2},
    {'id': '3', 'name': 'competitor_data',  'status': 'failed', 'pages':  3, 'items':  56,  'duration':  8.1,
     'error': 'HTTPError 403 Forbidden'},
]

report_root = create_report_xml(jobs_data)

# Pretty-print (Python 3.9+)
ET.indent(report_root, space='  ')
xml_output = ET.tostring(report_root, encoding='unicode', xml_declaration=False)
print(xml_output)

# Save to file
xml_path = OUTDIR / 'report.xml'
tree = ET.ElementTree(report_root)
tree.write(str(xml_path), encoding='UTF-8', xml_declaration=True)
print(f'\nSaved: {xml_path}  ({xml_path.stat().st_size} bytes)')

## 6. XML Namespaces

In [ ]:
ns_xml = """
<feed xmlns="http://www.w3.org/2005/Atom"
      xmlns:dc="http://purl.org/dc/elements/1.1/">
  <title>PyPath Blog</title>
  <entry>
    <title>Automating the Web with Python</title>
    <dc:creator>Alice Smith</dc:creator>
    <updated>2024-03-15T10:00:00Z</updated>
    <summary>Learn how to scrape and automate web tasks.</summary>
  </entry>
  <entry>
    <title>REST APIs in 30 Minutes</title>
    <dc:creator>Bob Jones</dc:creator>
    <updated>2024-03-10T09:00:00Z</updated>
    <summary>A hands-on intro to consuming REST APIs.</summary>
  </entry>
</feed>
"""

# Register namespaces so they appear cleanly in output
ATOM = 'http://www.w3.org/2005/Atom'
DC   = 'http://purl.org/dc/elements/1.1/'
ET.register_namespace('', ATOM)
ET.register_namespace('dc', DC)

feed = ET.fromstring(ns_xml)

print('Feed title:', feed.findtext(f'{{{ATOM}}}title'))
print()
for entry in feed.findall(f'{{{ATOM}}}entry'):
    title   = entry.findtext(f'{{{ATOM}}}title')
    creator = entry.findtext(f'{{{DC}}}creator')
    updated = entry.findtext(f'{{{ATOM}}}updated')
    print(f'  Title  : {title}')
    print(f'  Author : {creator}')
    print(f'  Updated: {updated}')
    print()

## 7. Format Comparison — Same Config in All Four Formats

Use this table to help you choose the right format:

| Format | Best for | Pros | Cons |
|---|---|---|---|
| **JSON** | APIs, web, data exchange | Universal, fast parsers, built-in | No comments, verbose nesting |
| **YAML** | Config files, CI/CD, K8s | Human-readable, comments, anchors | Indent-sensitive, security risks |
| **TOML** | Project config (`pyproject.toml`) | Clear, unambiguous, sections | Less tooling than YAML/JSON |
| **XML** | Enterprise, SOAP, RSS, SVG | Namespaces, schema validation | Verbose, complex to parse |

In [ ]:
# The same database config in all four formats

json_cfg = json.dumps({
    'database': {'host': 'localhost', 'port': 5432, 'name': 'pypath_db',
                 'pool': {'min': 2, 'max': 10}}
}, indent=2)

yaml_cfg = yaml.safe_dump({
    'database': {'host': 'localhost', 'port': 5432, 'name': 'pypath_db',
                 'pool': {'min': 2, 'max': 10}}
}, default_flow_style=False)

toml_cfg = """[database]\nhost = \"localhost\"\nport = 5432\nname = \"pypath_db\"\n\n[database.pool]\nmin = 2\nmax = 10"""

db_root = ET.Element('database')
ET.SubElement(db_root, 'host').text = 'localhost'
ET.SubElement(db_root, 'port').text = '5432'
ET.SubElement(db_root, 'name').text = 'pypath_db'
pool_el = ET.SubElement(db_root, 'pool')
ET.SubElement(pool_el, 'min').text = '2'
ET.SubElement(pool_el, 'max').text = '10'
ET.indent(db_root, space='  ')
xml_cfg = ET.tostring(db_root, encoding='unicode')

for fmt, text in [('JSON', json_cfg), ('YAML', yaml_cfg), ('TOML', toml_cfg), ('XML', xml_cfg)]:
    print(f'=== {fmt} ({len(text)} chars) ===')
    print(text)
    print()

## Practice Exercises

**Exercise 1 — YAML Config Merger**  
Write a function `merge_configs(base_path: str, override_path: str) -> dict` that loads two YAML files and deep-merges them (override values take precedence over base values for matching keys, but base-only keys are preserved). Save the merged result to a third YAML file.

**Exercise 2 — RSS Feed Parser**  
Fetch the RSS feed at `https://feeds.feedburner.com/PythonInsider` (or any valid RSS URL) using `requests`, parse it with `xml.etree.ElementTree`, and print the title and published date of the 5 most recent entries. Handle the Atom namespace correctly.

**Exercise 3 — Config Format Converter**  
Write a CLI-style function `convert_config(src_path: str, dst_format: str) -> str` that:
- Reads a YAML or JSON file from `src_path`
- Returns the data serialized in the specified `dst_format` ('json', 'yaml', or 'toml' using `tomli-w`)
- Preserves key ordering and handles nested structures
- Raises `ValueError` for unsupported formats